# Self-Supervised SimCLR (Unexecuted)

Portfolio edition: Behrad Mazaheri. Template/source credit: Source header: Dr. Soleymani, Spring 2026.

**Static review only — not rerun.** Outputs below are retained from `P1_completed.ipynb`, including failures or missing executions. They do not validate later revisions. Read README and REVIEW before running cells or citing metrics. Source cell references use zero-based indices before this added banner.


<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
    <p><b>Practical Homework 5 — Self-Supervised Learning (SimCLR)</b></p>
</div>

---

**Full Name:**

**Student ID:**

# Self-Supervised Learning with SimCLR — Homework Notebook

In this assignment you will implement **SimCLR** (*A Simple Framework for Contrastive Learning of Visual Representations*) in PyTorch, step by step.

The notebook is split into 9 sections. Each section starts with a short explanation, occasionally asks a theoretical question, and then contains one or more **TODO** code cells for you to complete. Read the comment at the top of every TODO cell carefully — it tells you exactly what is expected.

## Outline

1. **Introduction** — what is contrastive learning and why it works.
2. **Setup** — imports, device configuration, helper functions.
3. **Data Augmentation** — the SimCLR augmentation pipeline.
4. **Dataset & DataLoader** — generating two views per unlabelled image.
5. **Model Architecture** — ResNet backbone + projection head.
6. **Contrastive Loss** — the NT-Xent (InfoNCE) loss.
7. **Training Loop** — putting everything together.
8. **Linear Evaluation** — freezing the encoder and training a linear classifier on top.
9. **Discussion** — analysing the results.

> **How to submit:** Run every cell from top to bottom in order, answer all theoretical questions in markdown cells right after the question, and make sure all TODOs are completed before exporting.

## 1. Introduction

Supervised deep learning requires massive labelled datasets, which are expensive and time-consuming to create.
Self-supervised learning (SSL) is a paradigm where a model learns representations from the data itself, without external labels. One of the most successful SSL approaches is **contrastive learning**.

### The core idea

We take an image $x$ and create two random augmentations (views) $\tilde{x}_i$ and $\tilde{x}_j$.
These two views form a **positive pair** — they come from the same underlying image and should have similar feature representations.
All other images in the batch are treated as **negative examples**.

A neural network encoder $f(\cdot)$ followed by a small projection head $g(\cdot)$ maps each view to an embedding $z = g(f(\tilde{x}))$.
The loss function encourages embeddings of a positive pair to be close (similar) while pushing apart embeddings of negative examples.

### SimCLR in one paragraph

SimCLR (Chen et al., 2020) combines: (i) a strong **stochastic augmentation** module (random crop, colour jitter, random flip, Gaussian blur), (ii) a **base encoder** (ResNet-18/50), (iii) a small **projection head** (2-layer MLP), and (iv) a **contrastive loss** computed on the L2-normalised embeddings — trained with large batches and a cosine-annealing learning rate.

> **Question 1.** Why do we need a projection head $g(\cdot)$ on top of the encoder, instead of computing the loss directly on encoder features $h = f(\tilde{x})$?
>
> **Question 2.** What is the role of the temperature parameter $\tau$ in the NT-Xent loss?

**Your answer to Q1 (projection head):**

The projection head separates the representation needed by the contrastive objective from the representation kept for downstream tasks. The NT-Xent loss can encourage invariance to augmentations by discarding information such as colour or local detail. If the loss acts directly on \(h\), this information must be removed from the encoder output itself. With a nonlinear head, \(g\) can specialize \(z=g(h)\) for contrastive matching while \(h\) retains more useful information. Therefore downstream tasks normally use \(h\), not \(z\).

**Your answer to Q2 (temperature \(\tau\)):**

The temperature rescales every similarity logit before the softmax. A smaller \(\tau\) makes the softmax sharper and concentrates learning on the most similar, difficult negatives; a larger \(\tau\) produces a flatter distribution. Thus \(\tau\) controls both how strongly the model separates close examples and the scale of the gradients. If it is too small training may become unstable or overemphasize false negatives, while an overly large value gives weak discrimination.


## 2. Setup

Below is the only cell in this section. The imports are already filled in (so you can focus on the SimCLR-specific parts), but you must complete two small TODOs: setting a reproducibility seed and selecting the device.

In [ ]:
# -------------------- Setup --------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast

import numpy as np
import os
import sys
from tqdm import tqdm
import logging
from datetime import datetime

# Reproducibility.
torch.manual_seed(0)
np.random.seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

# Pick CUDA when available, otherwise CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


## 3. Data Augmentation

SimCLR uses a carefully designed augmentation pipeline that creates two *independent* random views of every image:

- **Random Resized Crop** — random crop + resize to a fixed size.
- **Random Horizontal Flip** — probability 0.5.
- **Colour Jitter** — random changes to brightness, contrast, saturation, hue (strength scaled by $s$).
- **Random Grayscale** — probability 0.2.
- **Gaussian Blur** — smooths the image with a random sigma.

### 3.1 Gaussian Blur

`torchvision` doesn't ship the SimCLR-style Gaussian blur, so we build it ourselves with two separable 1-D convolutions. Given kernel radius $r$ and sigma $\sigma$ sampled uniformly from $[0.1, 2.0]$, the 1-D weights are

$$ w_i \;\propto\; \exp\!\left(-\tfrac{i^2}{2\sigma^2}\right), \qquad i \in \{-r,\dots,r\}, $$

normalised to sum to 1. We then apply the horizontal kernel, then the vertical kernel.

> **Question 3.** Why is it important to randomly sample $\sigma$ per image rather than fixing it? What does this stochasticity buy us in contrastive learning?

**Your answer to Q3 (random \(\sigma\)):**

Sampling \(\sigma\) gives a continuum of blur strengths, so the positive pair cannot be matched by exploiting one fixed blur signature. The encoder must instead learn features that are stable under different amounts of loss of high-frequency detail. This increases augmentation diversity, reduces overfitting to the transformation pipeline, and usually improves invariance and generalization. A fixed \(\sigma\) would provide a narrower and more predictable pretext task.


In [ ]:
# -------------------- GaussianBlur --------------------
class GaussianBlur:
    """Apply a Gaussian blur with a random sigma to a PIL image (CPU)."""

    def __init__(self, kernel_size: int):
        # Force kernel_size to be odd so it has a well-defined centre.
        radius = kernel_size // 2
        kernel_size = radius * 2 + 1
        self.kernel_size = kernel_size
        self.radius = radius

        # Depthwise convolution applies the same spatial operation separately
        # to each of the three colour channels.
        self.blur_h = nn.Conv2d(
            3, 3, kernel_size=(kernel_size, 1),
            stride=1, padding=0, groups=3, bias=False
        )
        self.blur_v = nn.Conv2d(
            3, 3, kernel_size=(1, kernel_size),
            stride=1, padding=0, groups=3, bias=False
        )
        self.blur = nn.Sequential(
            nn.ReflectionPad2d(radius),
            self.blur_h,
            self.blur_v,
        )

        self.pil_to_tensor = transforms.ToTensor()
        self.tensor_to_pil = transforms.ToPILImage()

    def __call__(self, img):
        img_tensor = self.pil_to_tensor(img).unsqueeze(0)

        sigma = np.random.uniform(0.1, 2.0)
        x = np.arange(-self.radius, self.radius + 1, dtype=np.float32)
        x = np.exp(-(x ** 2) / (2 * sigma ** 2))
        x = x / x.sum()
        x = torch.from_numpy(x).view(1, -1).repeat(3, 1)

        self.blur_h.weight.data.copy_(x.view(3, 1, self.kernel_size, 1))
        self.blur_v.weight.data.copy_(x.view(3, 1, 1, self.kernel_size))

        with torch.no_grad():
            img_tensor = self.blur(img_tensor).squeeze(0)

        return self.tensor_to_pil(img_tensor)


### 3.2 SimCLR augmentation pipeline

Now we compose the full transform. The strength factor $s$ controls the colour jitter intensity (typically $s=1$ for ImageNet-scale data, $s=0.5$ for smaller datasets).

The recommended pipeline is, in order:

1. `RandomResizedCrop(size)`
2. `RandomHorizontalFlip()`
3. `ColorJitter(0.8s, 0.8s, 0.8s, 0.2s)` wrapped in `RandomApply(p=0.8)`
4. `RandomGrayscale(p=0.2)`
5. Your custom `GaussianBlur(kernel_size = int(0.1 * size))`
6. `ToTensor()`

In [ ]:
# -------------------- SimCLR transform --------------------
def get_simclr_pipeline_transform(size: int, s: float = 1.0):
    """Return a torchvision `Compose` implementing the SimCLR augmentation."""
    color_jitter = transforms.ColorJitter(
        0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s
    )
    data_transforms = transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        GaussianBlur(kernel_size=int(0.1 * size)),
        transforms.ToTensor(),
    ])
    return data_transforms


### 3.3 Contrastive-learning view generator

For every image we need **two** augmented views (the positive pair). We wrap the transform in a tiny class whose `__call__` applies the base transform `n_views` times to the *same* PIL input and returns a list of tensors.

In [ ]:
# -------------------- ContrastiveLearningViewGenerator --------------------
class ContrastiveLearningViewGenerator:
    """Produce `n_views` independently augmented views of one PIL image."""

    def __init__(self, base_transform, n_views: int = 2):
        self.base_transform = base_transform
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transform(x) for _ in range(self.n_views)]


> **Question 4.** Why must the two views be produced by **two independent** random draws of the augmentation? What would go wrong if the *same* deterministic transform were applied to both views?

**Your answer to Q4:**

Independent draws change crop, colour, blur and other nuisance factors while preserving the underlying image identity. To bring those views together, the network must learn invariance to the transformations. If both branches received exactly the same deterministic pixels, the positive-matching task would be trivial: the model could match low-level texture or even nearly identical pixel patterns. It would have little reason to learn semantic, transformation-invariant features, so downstream transfer would be weaker.


## 4. Dataset and DataLoader

SimCLR does not need labels, so we use either **STL-10** (in `split='unlabeled'` mode, 96×96 images) or **CIFAR-10** (32×32 images, labels ignored).

The dataset wraps the `ContrastiveLearningViewGenerator` as its `transform`, so each sample returns `(views, _)` where `views` is a list of `n_views` tensors of shape `[C, H, W]`. The label is ignored.

We use a large batch (e.g. 256 or 512) because SimCLR benefits from many negatives, and `drop_last=True` to keep all batches the same size.

> **Question 5.** Why do we set `drop_last=True` in the `DataLoader`? What could go wrong with a final, smaller batch?

In [ ]:
# -------------------- Dataset setup --------------------
class ContrastiveLearningDataset:
    def __init__(self, root_folder: str):
        self.root_folder = root_folder

    @staticmethod
    def get_simclr_pipeline_transform(size: int, s: float = 1.0):
        return get_simclr_pipeline_transform(size, s)

    def get_dataset(self, name: str, n_views: int):
        valid_datasets = {
            'cifar10': lambda: datasets.CIFAR10(
                self.root_folder, train=True, download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(32), n_views
                ),
            ),
            'stl10': lambda: datasets.STL10(
                self.root_folder, split='unlabeled', download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(96), n_views
                ),
            ),
        }
        if name not in valid_datasets:
            raise ValueError(f"Invalid dataset name: {name}. Choose 'stl10' or 'cifar10'.")
        return valid_datasets[name]()


dataset_name = 'stl10'          # or 'cifar10'
batch_size = 256
n_views = 2

dataset = ContrastiveLearningDataset('./data')
train_dataset = dataset.get_dataset(dataset_name, n_views)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

print(f"Number of training batches: {len(train_loader)}")
print(
    f"Each batch returns a list of {n_views} views, each of shape:",
    next(iter(train_loader))[0][0].shape,
)


**Your answer to Q5 (`drop_last=True`):**

The contrastive mask and the number of logits are determined by the current batch size, so the loss function used here can technically process a smaller last batch. Nevertheless, `drop_last=True` keeps the number of negatives and tensor shapes constant across iterations. This avoids a noisy final update with far fewer negatives, makes batch-normalization statistics more consistent, and is essential in implementations that cache masks or assume a configured fixed batch size.


## 5. Model Architecture

SimCLR uses a standard ResNet backbone as the encoder $f(\cdot)$ and a small MLP as the projection head $g(\cdot)$. The projection head maps the encoder output (e.g. 512-d for ResNet-18) into a lower-dimensional embedding space (e.g. 128-d). The whole model is simply $g \circ f$.

You will implement `ResNetSimCLR`:

- Take a base model name (`'resnet18'` or `'resnet50'`) and instantiate it from `torchvision.models` **without** ImageNet weights.
- Read the encoder's feature dimension from `backbone.fc.in_features` (call it `dim_mlp`).
- **Replace** `backbone.fc` with a 2-layer projection head:
  $$\texttt{Linear}(\text{dim\_mlp} \to \text{dim\_mlp}) \to \texttt{ReLU} \to \texttt{Linear}(\text{dim\_mlp} \to \text{out\_dim})$$

After contrastive pre-training the projection head is discarded; only the encoder is kept for downstream tasks.

> **Question 6.** Why does the projection head use the same dimension for its hidden layer as the encoder output? (Hint: capacity vs. bottleneck.)

**Your answer to Q6 (projection-head hidden dimension):**

Keeping the hidden layer at the encoder dimension gives the nonlinear head enough capacity to reorganize the full representation before the final \(128\)-dimensional contrastive bottleneck. If the first layer immediately compressed \(h\), useful information could be discarded too early and the head would have less freedom to absorb contrastive-objective-specific transformations. The final layer still supplies the desired compact embedding, so this design balances capacity with a controlled output size.


In [ ]:
# -------------------- ResNetSimCLR --------------------
class ResNetSimCLR(nn.Module):
    def __init__(self, base_model: str = 'resnet18', out_dim: int = 128):
        super().__init__()
        self.backbone = self._get_basemodel(base_model)
        dim_mlp = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(inplace=True),
            nn.Linear(dim_mlp, out_dim),
        )

    def _get_basemodel(self, model_name: str):
        resnet_models = {
            'resnet18': models.resnet18(weights=None, num_classes=1000),
            'resnet50': models.resnet50(weights=None, num_classes=1000),
        }
        if model_name not in resnet_models:
            raise ValueError(f"Invalid backbone: {model_name}. Choose 'resnet18' or 'resnet50'.")
        return resnet_models[model_name]

    def forward(self, x):
        return self.backbone(x)


model = ResNetSimCLR(base_model='resnet18', out_dim=128).to(device)
dummy = torch.randn(2, 3, 96, 96).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")


> **Question 7.** When we evaluate downstream by freezing the encoder and training only a linear classifier on top, what happens to gradients through the (now removed) projection head? Why is throwing the head away a *reasonable* choice rather than a wasteful one?

**Your answer to Q7:**

Once the projection head is removed, it is absent from the downstream computation graph, so it receives no gradients and its parameters are not used. The encoder is also frozen during linear evaluation; gradients are needed only for the new classifier. This is reasonable because the head was an auxiliary training module specialized for NT-Xent, whereas the encoder feature \(h\) is deliberately preserved as the transferable representation. The head has already served its purpose by shaping how the encoder was trained.


## 6. Contrastive Loss — NT-Xent

SimCLR uses the **Normalised Temperature-scaled Cross Entropy** (NT-Xent / InfoNCE) loss.

For a batch of $N$ images we obtain $2N$ embeddings (two views per image). Let $z_i, z_j$ be a positive pair and $\tau$ the temperature. The cosine similarity scaled by $1/\tau$ is

$$ \mathrm{sim}(z_a, z_b) \;=\; \frac{1}{\tau}\, \frac{z_a^\top z_b}{\lVert z_a\rVert \,\lVert z_b\rVert}. $$

For each positive pair $(i,j)$:

$$ \ell(i,j) \;=\; -\log \frac{\exp(\mathrm{sim}(z_i, z_j))}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k))}. $$

The total loss is averaged over all $2N$ such terms.

In code we build the full $2N \times 2N$ similarity matrix, drop the diagonal (self-similarities), and reshape so each row has 1 positive logit + $(2N-2)$ negative logits. With the positive always at index 0 the loss reduces to plain `CrossEntropyLoss` against the target label `0`.

> **Question 8.** Show that formulating this as cross-entropy over $(2N-1)$ classes — 1 positive + $(2N-2)$ negatives — with the correct class always being the positive, is equivalent to the equation above. *Hint:* expand $-\log \mathrm{softmax}_0(\cdot)$.

**Your answer to Q8 (cross-entropy reformulation):**

For anchor \(i\), arrange the candidate logits so that
\[
a_0=\operatorname{sim}(z_i,z_j)
\]
is the positive logit and \(a_1,\ldots,a_{2N-2}\) are similarities to all other non-self samples. Cross-entropy with target class \(0\) is
\[
\begin{aligned}
\operatorname{CE}(a,0)
&=-\log\left(\operatorname{softmax}(a)_0\right)\\
&=-\log\frac{\exp(a_0)}
{\sum_{m=0}^{2N-2}\exp(a_m)}\\
&=-\log\frac{\exp(\operatorname{sim}(z_i,z_j))}
{\sum_{k\ne i}\exp(\operatorname{sim}(z_i,z_k))}.
\end{aligned}
\]
The denominator contains the positive and all \(2N-2\) negatives, but excludes the self-similarity \(k=i\). This is exactly \(\ell(i,j)\). Averaging this cross-entropy over the \(2N\) anchors gives the NT-Xent loss.


In [ ]:
# -------------------- info_nce_loss --------------------
def info_nce_loss(features, temperature: float = 0.07):
    """
    Args:
        features: Tensor of shape [2 * batch_size, out_dim].
        temperature: positive scalar tau.

    Returns:
        logits: [2N, 2N-1], with the positive in column 0.
        labels: [2N], all zeros.
    """
    if features.ndim != 2 or features.shape[0] % 2 != 0:
        raise ValueError("features must have shape [2N, out_dim].")
    if temperature <= 0:
        raise ValueError("temperature must be positive.")

    batch_size = features.shape[0] // 2
    device = features.device

    sample_ids = torch.arange(batch_size, device=device).repeat(2)
    labels = sample_ids.unsqueeze(0).eq(sample_ids.unsqueeze(1))

    features = F.normalize(features, dim=1)
    similarity_matrix = features @ features.T

    mask = torch.eye(labels.shape[0], dtype=torch.bool, device=device)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(
        similarity_matrix.shape[0], -1
    )

    positives = similarity_matrix[labels].view(labels.shape[0], 1)
    negatives = similarity_matrix[~labels].view(labels.shape[0], -1)
    logits = torch.cat([positives, negatives], dim=1) / temperature

    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
    return logits, labels


> **Question 9.** What is the effect of *lowering* the temperature \(\tau\)? Discuss how it changes (i) the concentration of the softmax distribution and (ii) the magnitude of the gradients with respect to hard negatives.

**Your answer to Q9:**

Lowering \(\tau\) divides logits by a smaller number, increasing their separation. Therefore the softmax becomes more concentrated on the largest logits. For cross-entropy,
\[
\frac{\partial L}{\partial s_k}=\frac{p_k-y_k}{\tau},
\]
where \(s_k\) is an unscaled similarity. The explicit \(1/\tau\) factor increases gradient scale, while the sharper \(p_k\) assigns most negative-side gradient to the most similar hard negatives. Easy negatives receive almost no probability and little gradient. An excessively small temperature can consequently produce large, high-variance updates and can strongly repel semantically related false negatives.


## 7. Training Loop

We now glue everything together. Per batch:

1. Concatenate the two views along dim 0 → tensor of shape `[2N, C, H, W]`.
2. Forward through the model → embeddings of shape `[2N, out_dim]`.
3. Compute `logits, labels = info_nce_loss(features, tau)`.
4. Apply `nn.CrossEntropyLoss` to obtain the scalar loss.
5. Backprop with `GradScaler` (mixed precision is optional).
6. Log loss and top-1/top-5 contrastive accuracy every `log_every_n_steps`.
7. Step the cosine-annealing scheduler after each epoch, after a 10-epoch warm-up.

You will fill in three code cells:

- **Cell 28** — the `SimCLR` trainer class.
- **Cell 29** — an `accuracy(...)` helper for top-k accuracy.
- **Cell 30** — the hyperparameter dict + actual training run.

In [ ]:
# -------------------- SimCLR Trainer --------------------
class SimCLR:
    def __init__(self, model, optimizer, scheduler, args):
        self.model = model.to(args['device'])
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.args = args
        self.criterion = nn.CrossEntropyLoss().to(args['device'])
        self.scaler = GradScaler(enabled=args['fp16_precision'])

        os.makedirs('./logs', exist_ok=True)
        logging.basicConfig(
            filename=os.path.join('./logs', f'training_{datetime.now():%Y%m%d_%H%M%S}.log'),
            level=logging.DEBUG,
        )
        logging.info("SimCLR training started.")

    def info_nce_loss(self, features):
        return info_nce_loss(features, self.args['temperature'])

    def train(self, train_loader):
        n_iter = 0
        logging.info(f"Training for {self.args['epochs']} epochs.")
        self.model.train()

        for epoch_counter in range(self.args['epochs']):
            pbar = tqdm(train_loader, desc=f"Epoch {epoch_counter + 1}")
            for images, _ in pbar:
                images = torch.cat(images, dim=0).to(
                    self.args['device'], non_blocking=True
                )

                self.optimizer.zero_grad(set_to_none=True)
                with autocast(enabled=self.args['fp16_precision']):
                    features = self.model(images)
                    logits, labels = self.info_nce_loss(features)
                    loss = self.criterion(logits, labels)

                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()

                if n_iter % self.args['log_every_n_steps'] == 0:
                    top1, top5 = accuracy(logits, labels, topk=(1, 5))
                    pbar.set_postfix({
                        'Loss': f'{loss.item():.4f}',
                        'Top1': f'{top1.item():.2f}%',
                        'Top5': f'{top5.item():.2f}%',
                    })
                    logging.debug(
                        f"Iter {n_iter} | Loss {loss.item():.4f} | "
                        f"Top1 {top1.item():.2f}%"
                    )
                n_iter += 1

            if epoch_counter >= 10:
                self.scheduler.step()

        logging.info("Training finished.")
        torch.save({
            'epoch': self.args['epochs'],
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }, 'simclr_checkpoint.pth')
        print("Checkpoint saved to simclr_checkpoint.pth")


In [ ]:
# -------------------- Top-k accuracy helper --------------------
def accuracy(output, target, topk=(1,)):
    """Compute the top-k accuracy (as percentages) for each k in `topk`."""
    with torch.no_grad():
        maxk = min(max(topk), output.size(1))
        batch_size = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))

        res = []
        for k in topk:
            effective_k = min(k, output.size(1))
            correct_k = correct[:effective_k].any(dim=0).float().sum()
            res.append(correct_k.mul(100.0 / batch_size))
        return res


In [ ]:
# -------------------- Training configuration and execution --------------------
args = {
    'device': device,
    'epochs': 10,
    'batch_size': batch_size,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'temperature': 0.07,
    'fp16_precision': False,
    'log_every_n_steps': 50,
    'out_dim': 128,
    'n_views': 2,
}

model = ResNetSimCLR(
    base_model='resnet18', out_dim=args['out_dim']
).to(device)
optimizer = optim.Adam(
    model.parameters(),
    lr=args['lr'],
    weight_decay=args['weight_decay'],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, args['epochs'] - 10),
    eta_min=0,
    last_epoch=-1,
)

simclr = SimCLR(model, optimizer, scheduler, args)
# Uncomment the next line to perform contrastive pre-training.
# simclr.train(train_loader)


> **Question 10.** Why is a *cosine-annealing* learning rate schedule used (after a short warm-up) instead of a constant LR? What would change qualitatively in the loss curves if we kept LR constant?

**Your answer to Q10:**

Warm-up avoids large, unstable updates while randomly initialized features and batch-normalization statistics are still changing rapidly. Cosine annealing then decreases the learning rate smoothly: larger early updates explore and learn broad structure, whereas smaller late updates refine the representation without abrupt schedule changes. With a constant learning rate, the loss would generally fall at first but fluctuate around a noisier plateau and may fail to settle into as good a minimum. A constant LR small enough to be stable late in training would instead make early progress unnecessarily slow.


## 8. Linear Evaluation

To assess the *quality* of the learned representations we follow the standard **linear-evaluation protocol**:

1. Load the pre-trained SimCLR checkpoint (`simclr_checkpoint.pth`).
2. Discard the projection head and bolt on a fresh **linear classifier** that maps the encoder output to `num_classes`.
3. **Freeze** all parameters of the backbone — only the new linear layer is trained.
4. Train on the labelled training split (STL-10 `train` or CIFAR-10) with `CrossEntropyLoss`.
5. Report top-1 and top-5 accuracy on the held-out test set.

High accuracy with a *frozen* encoder indicates that the contrastive objective produced semantically useful features.

In [ ]:
# -------------------- Linear Evaluation --------------------
if dataset_name == 'stl10':
    train_data = datasets.STL10('./data', split='train', download=True,
                                transform=transforms.ToTensor())
    test_data = datasets.STL10('./data', split='test', download=True,
                               transform=transforms.ToTensor())
    num_classes = 10
elif dataset_name == 'cifar10':
    train_data = datasets.CIFAR10('./data', train=True, download=True,
                                  transform=transforms.ToTensor())
    test_data = datasets.CIFAR10('./data', train=False, download=True,
                                 transform=transforms.ToTensor())
    num_classes = 10
else:
    raise ValueError("Unsupported dataset for evaluation")

eval_train_loader = DataLoader(
    train_data, batch_size=256, shuffle=True, num_workers=2,
    pin_memory=torch.cuda.is_available()
)
eval_test_loader = DataLoader(
    test_data, batch_size=512, shuffle=False, num_workers=2,
    pin_memory=torch.cuda.is_available()
)


def load_model_for_evaluation(checkpoint_path, base_model='resnet18', num_classes=10):
    model = ResNetSimCLR(base_model=base_model, out_dim=128)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['state_dict']

    for k in list(state_dict.keys()):
        if k.startswith('backbone.fc'):
            del state_dict[k]

    log = model.load_state_dict(state_dict, strict=False)
    print("Missing keys when loading backbone:", log.missing_keys)

    dim_mlp = model.backbone.fc[0].in_features
    model.backbone.fc = nn.Linear(dim_mlp, num_classes)
    return model.to(device)


eval_model = load_model_for_evaluation(
    'simclr_checkpoint.pth', 'resnet18', num_classes
)

for name, param in eval_model.named_parameters():
    param.requires_grad = 'backbone.fc' in name

trainable_params = [n for n, p in eval_model.named_parameters() if p.requires_grad]
print("Trainable parameters:", trainable_params)

eval_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, eval_model.parameters()),
    lr=1e-3,
    weight_decay=1e-6,
)
eval_criterion = nn.CrossEntropyLoss()

eval_epochs = 20
for epoch in range(eval_epochs):
    # Keep the frozen encoder, including its BatchNorm running statistics,
    # fixed. The linear layer still receives gradients in evaluation mode.
    eval_model.eval()
    running_loss = 0.0
    seen = 0
    for images, labels in eval_train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        eval_optimizer.zero_grad(set_to_none=True)
        logits = eval_model(images)
        loss = eval_criterion(logits, labels)
        loss.backward()
        eval_optimizer.step()

        running_loss += loss.item() * images.size(0)
        seen += images.size(0)

    total = 0
    top1_correct = 0
    top5_correct = 0
    with torch.no_grad():
        for images, labels in eval_test_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = eval_model(images)
            k = min(5, logits.size(1))
            pred = logits.topk(k, dim=1).indices
            top1_correct += pred[:, 0].eq(labels).sum().item()
            top5_correct += pred.eq(labels.unsqueeze(1)).any(dim=1).sum().item()
            total += labels.size(0)

    mean_loss = running_loss / seen
    test_top1 = 100.0 * top1_correct / total
    test_top5 = 100.0 * top5_correct / total
    print(
        f"Epoch {epoch + 1}: Loss={mean_loss:.4f} | "
        f"Test Top1={test_top1:.2f}% | Test Top5={test_top5:.2f}%"
    )


> **Question 11.** Why do we *freeze* the backbone during linear evaluation? What would happen — methodologically and empirically — if we instead fine-tuned the whole model end-to-end with the labels?

**Your answer to Q11:**

Freezing the backbone makes the classifier a controlled probe: its accuracy measures how linearly accessible class information already is in the self-supervised representation. Fine-tuning would allow labels to change the representation, so the result would measure a different protocol—the usefulness of the pre-trained model as an initialization—not representation quality alone. End-to-end fine-tuning often improves final accuracy, especially with enough labels, but it can overfit small labelled sets and obscures whether performance came from SSL features or supervised adaptation.


## 9. Discussion

### Batch size

With \(N\) images and two views, each anchor has \(2N-2\) in-batch negatives. Increasing the batch size therefore supplies a richer comparison set and usually improves the contrastive signal, but it also increases memory and computation and can require learning-rate retuning. Returns diminish once the batch already represents the data distribution well; additional negatives are then increasingly redundant, and false negatives (different images from the same semantic class) may become more common.

### Projection head

The head provides a space in which the model can discard augmentation-specific information to satisfy NT-Xent, without forcing the encoder representation \(h\) to discard the same information. It is discarded downstream because \(h\) is typically more general and information-rich, while \(z\) is specialized for the pre-training objective.

### Augmentation ablation

This comparison requires executed runs; no numerical result should be invented before running them. Use the same seed, split, epochs and optimizer for the full pipeline and the ablated pipeline, changing only one augmentation. Report:

| Run | Linear-eval top-1 |
|---|---:|
| Full SimCLR pipeline | **[fill after run]** |
| Without colour jitter *or* without Gaussian blur | **[fill after run]** |
| Change (ablated − full) | **[fill after run]** |

Interpretation: the augmentation whose removal causes the larger accuracy decrease is more important for this dataset. Colour jitter is often particularly important because it prevents colour-histogram shortcuts, whereas blur discourages dependence on fine texture. The actual conclusion must follow the measured accuracies.

### Random baseline

Train the same frozen linear classifier on a randomly initialized ResNet using identical labelled data and optimization. Report:

| Backbone | Linear-eval top-1 |
|---|---:|
| SimCLR-pretrained | **[fill after run]** |
| Randomly initialized | **[fill after run]** |
| SimCLR gain | **[fill after run] percentage points** |

A positive gap isolates the value of contrastive pre-training. The random backbone may exceed chance because a linear combination of random convolutional features can still contain weak signal, but it should normally be well below a successfully pre-trained encoder.

### Longer training

The supplied configuration uses only 10 epochs and is a functional demonstration, not the full SimCLR regime. Record the 10-epoch result and a 100+-epoch result under the same evaluation protocol:

| Pre-training length | Linear-eval top-1 |
|---|---:|
| 10 epochs | **[fill after run]** |
| 100+ epochs | **[fill after run]** |

Longer training should generally lower contrastive loss and improve linear accuracy until saturation, although the outcome also depends on augmentation strength, batch size and the learning-rate schedule.

### Further Reading

- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for Contrastive Learning of Visual Representations*. ICML 2020.
- [SimCLR — official TF repo](https://github.com/google-research/simclr)
- [PyTorch SimCLR re-implementation](https://github.com/sthalles/SimCLR)
- [Exploring SimCLR — blog post](https://sthalles.github.io/simple-self-supervised-learning/)
